# multilingual-e5-large — SCCL (Supporting Clustering with Contrastive Learning)

The newest and most compute-intensive method tried in this project so far. Unlike every other
notebook in `Aggregation_Pipeline/clustering/`, which treats `multilingual-e5-large` as a frozen
feature extractor (embed once, cluster the fixed embeddings), **SCCL fine-tunes the backbone
itself**, jointly on two losses:

1. **Contrastive (Instance-CL) loss** — pulls together the representations of two augmented views
   of the *same* article while pushing apart every other article in the batch (InfoNCE / NT-Xent).
   This directly targets the problem the SCCL paper identifies: early in training (or, here, before
   any clustering-specific fine-tuning at all), different true event-categories often overlap
   heavily in representation space, which hurts every distance-based clustering method used
   elsewhere in this project.
2. **Clustering loss** — the same DEC/IDEC-style soft-assignment (Student\'s-t kernel) +
   self-sharpened target distribution + KL divergence used in
   `deep_clustering/clustering_e5_DEC.ipynb`, computed on top of the being-fine-tuned
   representation instead of a frozen autoencoder\'s latent space.

**Deviation from the SCCL paper, and why**: the original method augments text via back-translation
or synonym replacement, both of which need language resources this environment doesn\'t have for
Sinhala (no reliable Sinhala synonym dictionary, and back-translation would require a separate
Sinhala↔X MT model plus its own compute/API budget). Instead this notebook uses two
resource-free augmentations that require no Sinhala-specific tooling:
- **random word deletion** (drop each word independently with some probability)
- **random adjacent word swap** (locally reorder a fraction of word pairs)

This is a legitimate simplification, not a fundamentally different method — it changes *how* the
two augmented views are produced, not the two-loss training objective that defines SCCL.

**Compute expectations**: this fine-tunes all ~560M parameters of `multilingual-e5-large` with
gradients enabled (every other notebook here runs it frozen, under `torch.no_grad()`). Expect this
to be by far the slowest notebook in this project — small batch size and a short max sequence
length are used deliberately to keep it within a single Kaggle T4\'s 16GB VRAM.

In [ ]:
!pip install -q umap-learn

In [ ]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
import random
from pathlib import Path

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_sccl")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

In [ ]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

In [ ]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

In [ ]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

In [ ]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

In [ ]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

In [ ]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

In [ ]:
# Build documents (title + body); the "passage: " prefix e5 needs is added at tokenization
# time in embed_passages() below, AFTER augmentation, so the prefix itself is never augmented away
def build_document(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    return f"{title}. {body}" if title else body


df["document"] = df.apply(lambda row: build_document(row["title"], row["text"]), axis=1)
documents = df["document"].tolist()
print(f"Documents: {len(documents)}")
documents[0][:300]

In [ ]:
# Resource-free text augmentation: random word deletion + random adjacent word swap
# (see intro cell for why this replaces SCCL\'s original back-translation/synonym augmentation)
def augment_text(text, delete_prob=0.1, swap_prob=0.1, rng=None):
    rng = rng or random
    words = text.split()
    if not words:
        return text
    kept = [w for w in words if rng.random() > delete_prob]
    if not kept:
        kept = words[:1]
    for i in range(len(kept) - 1):
        if rng.random() < swap_prob:
            kept[i], kept[i + 1] = kept[i + 1], kept[i]
    return " ".join(kept)


sample_rng = random.Random(SEED)
print("original: ", documents[0][:200])
print("augmented:", augment_text(documents[0][:200], rng=sample_rng))

In [ ]:
# Load multilingual-e5-large as a TRAINABLE model (no torch.no_grad() here — this is the
# one notebook in this project that fine-tunes the backbone itself)
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
MAX_LENGTH = 256  # shorter than the 512 used elsewhere in this project, to keep fine-tuning
                   # memory/time manageable on a single Kaggle GPU


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode_batch(texts, requires_grad):
    encoded = tokenizer(
        ["passage: " + t for t in texts],
        padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt",
    ).to(device)
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = F.normalize(pooled, p=2, dim=1)
    return pooled

In [ ]:
# Initial (pre-fine-tuning) embeddings for the whole corpus — used to (a) get a separation
# score comparable to every other notebook, and (b) pick k and initialize cluster centers before
# any fine-tuning starts
model.eval()
initial_embeddings = []
BATCH_SIZE = 16
for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i:i + BATCH_SIZE]
    initial_embeddings.append(encode_batch(batch, requires_grad=False).cpu())
initial_embeddings = torch.cat(initial_embeddings, dim=0).numpy()
print(initial_embeddings.shape)

In [ ]:
# Embedding separation score (comparable across notebooks): 1 - mean pairwise cosine similarity
# on a random 100-document sample of the raw (pre-fine-tuning) embeddings.
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(initial_embeddings), size=min(100, len(initial_embeddings)), replace=False)
sims = cosine_similarity(initial_embeddings[sample_idx])
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()
print(f"Separation score (pre-fine-tuning): {separation_score:.4f}")

In [ ]:
# Pick k and initialize cluster centers from the pre-fine-tuning embeddings (same coarse
# sweep used to seed k in the deep_clustering/ notebooks)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

candidate_k = list(range(10, 100, 15)) + list(range(100, 700, 100))
sweep_results = []
for k in candidate_k:
    if k >= len(initial_embeddings):
        continue
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(initial_embeddings)
    sil = silhouette_score(initial_embeddings, labels, metric="cosine")
    sweep_results.append((k, sil))

sweep_df = pd.DataFrame(sweep_results, columns=["k", "silhouette"]).sort_values("silhouette", ascending=False)
K = int(sweep_df.iloc[0]["k"])
print(f"k = {K}")

kmeans_init = KMeans(n_clusters=K, random_state=SEED, n_init=10).fit(initial_embeddings)
cluster_centers = nn.Parameter(
    torch.tensor(kmeans_init.cluster_centers_, dtype=torch.float32, device=device)
)

In [ ]:
# DEC-style soft assignment (Student\'s t) + self-sharpened target distribution — the
# clustering-loss half of SCCL\'s joint objective
def soft_assign(z, centers, alpha=1.0):
    dist_sq = torch.cdist(z, centers) ** 2
    numerator = (1.0 + dist_sq / alpha) ** (-(alpha + 1.0) / 2.0)
    return numerator / numerator.sum(dim=1, keepdim=True)


def target_distribution(q):
    weight = (q ** 2) / q.sum(dim=0)
    return (weight.t() / weight.sum(dim=1)).t()


def info_nce(z1, z2, temperature=0.5):
    z = torch.cat([z1, z2], dim=0)
    sim = z @ z.t() / temperature
    n = z1.size(0)
    sim.masked_fill_(torch.eye(2 * n, dtype=torch.bool, device=z.device), float("-inf"))
    positive_idx = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, positive_idx)

In [ ]:
# Joint fine-tuning: contrastive loss (on augmented pairs) + clustering loss (KL against a
# target distribution recomputed once per epoch over the FULL corpus using the current,
# being-fine-tuned model — same "retarget every epoch, hold fixed within it" scheme as
# deep_clustering/clustering_e5_DEC.ipynb, just with a trainable backbone instead of a frozen
# autoencoder latent space)
ETA_CLUSTERING = 1.0
N_EPOCHS = 5
optimizer = torch.optim.AdamW(list(model.parameters()) + [cluster_centers], lr=2e-5)

indices = list(range(len(documents)))

for epoch in range(N_EPOCHS):
    # Recompute the target distribution P over the whole corpus with the current model
    model.eval()
    with torch.no_grad():
        current_embeddings = []
        for i in range(0, len(documents), BATCH_SIZE):
            current_embeddings.append(encode_batch(documents[i:i + BATCH_SIZE], requires_grad=False))
        current_embeddings = torch.cat(current_embeddings, dim=0)
        q_full = soft_assign(current_embeddings, cluster_centers)
        p_full = target_distribution(q_full)

    model.train()
    random.shuffle(indices)
    epoch_contrastive, epoch_clustering, n_batches = 0.0, 0.0, 0
    for start in range(0, len(indices), BATCH_SIZE):
        batch_idx = indices[start:start + BATCH_SIZE]
        batch_texts = [documents[i] for i in batch_idx]

        aug1 = [augment_text(t) for t in batch_texts]
        aug2 = [augment_text(t) for t in batch_texts]
        z1 = encode_batch(aug1, requires_grad=True)
        z2 = encode_batch(aug2, requires_grad=True)
        contrastive_loss = info_nce(z1, z2)

        z_orig = encode_batch(batch_texts, requires_grad=True)
        q_batch = soft_assign(z_orig, cluster_centers)
        p_batch = p_full[batch_idx]
        clustering_loss = F.kl_div(q_batch.log(), p_batch, reduction="batchmean")

        loss = contrastive_loss + ETA_CLUSTERING * clustering_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_contrastive += contrastive_loss.item()
        epoch_clustering += clustering_loss.item()
        n_batches += 1

    print(f"epoch {epoch + 1}/{N_EPOCHS}  contrastive={epoch_contrastive / n_batches:.4f}  "
          f"clustering={epoch_clustering / n_batches:.4f}")

In [ ]:
# Final embeddings + cluster assignment from the fine-tuned model
model.eval()
with torch.no_grad():
    final_embeddings = []
    for i in range(0, len(documents), BATCH_SIZE):
        final_embeddings.append(encode_batch(documents[i:i + BATCH_SIZE], requires_grad=False))
    final_embeddings = torch.cat(final_embeddings, dim=0)
    q_final = soft_assign(final_embeddings, cluster_centers)
    final_labels = q_final.argmax(dim=1).cpu().numpy()
    final_latent = final_embeddings.cpu().numpy()

print(pd.Series(final_labels).value_counts().head(20))
print("Number of non-empty clusters:", len(set(final_labels)))

In [ ]:
# Clustering evaluation
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

n_clusters = len(set(final_labels))
sil = silhouette_score(final_latent, final_labels, metric="cosine")
dbi = davies_bouldin_score(final_latent, final_labels)
ch = calinski_harabasz_score(final_latent, final_labels)

print(f"Model: {embedding_model_name} + SCCL (contrastive + clustering fine-tuning)")
print(f"Articles: {len(df)} | Clusters: {n_clusters} | Noise ratio: 0.00% (n/a for this algorithm)")
print(f"Silhouette Score (cosine): {sil:.4f}")
print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")

scores_path = RESULTS_DIR / "e5_sccl_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "SCCL",
    "embedding_dim": final_latent.shape[1],
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": 0.0,
    "silhouette": round(sil, 4),
    "davies_bouldin": round(dbi, 4),
    "calinski_harabasz": round(ch, 2),
    "k": K,
    "n_epochs": N_EPOCHS,
    "eta_clustering": ETA_CLUSTERING,
}
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

In [ ]:
# Inspect sample titles per cluster (first few)
df["cluster_id"] = final_labels
for cluster_id, group in list(df.groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

In [ ]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

In [ ]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_sccl_assignments.csv"
df.drop(columns=["document"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")